In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
new_df = pd.read_csv("../../data/processed/new_churn.csv")
new_df.head()

,고객ID,API_ratio_mean,신규고객_최근구매경과일_R,신규고객_총구매횟수_F,신규고객_총구매금액_M,상품군_라벨인코딩_신규고객이탈율,멤버십_라벨인코딩_신규고객이탈율,연령대_라벨인코딩_신규고객이탈율,구매연도_라벨인코딩_신규고객이탈율,구매월_라벨인코딩_신규고객이탈율,신규고객이탈여부
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,68.666667,17,21,0.648983,0.004282,0.004277,0.003670,0.004088,0.004735,0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,29.818182,76,86,2.601932,0.003645,0.003641,0.003454,0.003985,0.003488,0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,121.000000,7,18,0.704780,0.003538,0.003536,0.003455,0.004374,0.004205,0
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.000000,471,2,0.060983,0.002761,0.002771,0.004165,0.003528,0.002469,0
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,134.000000,41,13,0.469695,0.003506,0.003515,0.004149,0.004229,0.003679,0


In [26]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE


# target = "신규고객이탈여부"
y = new_df['신규고객이탈여부']
X = new_df.drop(columns=["고객ID","신규고객이탈여부"]) # 가입일과 이탈일은 모델링에 사용하지 않음

# drop_cols = ["고객ID"]
# X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors="ignore") # 제거할 열이 존재하지 않을 경우 오류 방지


print(f'만들어진 샘플 비율 : {np.bincount(y)}') # 균형이 맞지 않음

smote = SMOTE(random_state =42)
X_resample, y_resample = smote.fit_resample(X,y)

print(f'resample 후 샘플 비율 : {np.bincount(y_resample)}')  # 균형이 잡힘



만들어진 샘플 비율 : [1226085  130142]
resample 후 샘플 비율 : [1226085 1226085]


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# 오버샘플링 전 데이터 활용

X_train, X_test, y_train, y_test = train_test_split(X_resample, y_resample, test_size=0.2, random_state=42)

rf_clf = XGBClassifier(random_state=0)
rf_clf.fit(X_train, y_train)

y_pred = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    245287
           1       1.00      1.00      1.00    245147

    accuracy                           1.00    490434
   macro avg       1.00      1.00      1.00    490434
weighted avg       1.00      1.00      1.00    490434



In [32]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import numpy as np

best = (None, 0)

num_cols = X_train.select_dtypes(include=[np.number]).columns

for c in num_cols:
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X_train[[c]], y_train)
    acc = accuracy_score(y_test, stump.predict(X_test[[c]]))
    if acc > best[1]:
        best = (c, acc)

print("한 개 피처만으로 나온 최고 정확도:", best)

한 개 피처만으로 나온 최고 정확도: ('신규고객_총구매횟수_F', 1.0)


In [ ]:
from xgboost import XGBClassifier


# xgb_clf = XGBClassifier( # 하이퍼파라미터 튜닝
#     # n_estimators=100,       # 트리의 개수       
#     # max_depth=6,            # 트리의 최대 깊이
#     # learning_rate=0.1, 
#     # random_state=0
# )
# xgb_clf.fit(X_train, y_train)


xgb_clf = XGBClassifier(random_state=0)
xgb_clf.fit(X_train, y_train)

y_pred_test = xgb_clf.predict(X_test)

print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    245218
           1       1.00      1.00      1.00     26028

    accuracy                           1.00    271246
   macro avg       1.00      1.00      1.00    271246
weighted avg       1.00      1.00      1.00    271246



In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_train = xgb_clf.predict(X_train)
y_pred_test = xgb_clf.predict(X_test)

print(accuracy_score(y_train, y_pred_train))
print(accuracy_score(y_test, y_pred_test))

print(classification_report(y_test, y_pred_test))



1.0
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    245218
           1       1.00      1.00      1.00     26028

    accuracy                           1.00    271246
   macro avg       1.00      1.00      1.00    271246
weighted avg       1.00      1.00      1.00    271246

